# 03 · Contagem de regimes (Tabela 1)

Lê `dados/processados/base_diaria.csv` (gerada por `02_processamento.ipynb`) e grava `resultados/contagem_regimes.csv`: dias, % da amostra e episódios de cada regime de crise. É a base da Tabela 1 do artigo.


In [1]:
from pathlib import Path

import pandas as pd

RAIZ = Path.cwd().parent if Path.cwd().name == "codigo" else Path.cwd()
PASTA_RESULTADOS = RAIZ / "resultados"

base = pd.read_csv(RAIZ / "dados" / "processados" / "base_diaria.csv", parse_dates=["data"], index_col="data")
print(f"Base: {base.index.min().date()} a {base.index.max().date()}, {len(base)} dias")

Base: 2003-09-16 a 2026-09-30, 5526 dias


## 2. Contagem de dias e episódios por regime

Um **episódio** é uma sequência de dias consecutivos (no calendário comum) dentro do regime. O número de episódios mostra se os dias de um regime vêm de muitos eventos ou de poucos eventos longos, o que importa para a P4: correlações de dias do mesmo episódio não são independentes.

In [2]:
CONTAGEM = {
    "crise_global": "VIX > 30",
    "crise_global_p90": "VIX > percentil 90",
    "crise_local_br": "Drawdown do Ibovespa ≤ −20%",
    "crise_eua": "Drawdown do S&P 500 ≤ −20%",
    "crise_p1_eua": "Crise EUA para a P1 (VIX > 30 ou drawdown do S&P 500)",
    "normal": "Normal",
    "so_global": "Só crise global",
    "so_local": "Só crise local",
    "ambas": "Crise local e global",
    "so_local_p90": "Só crise local (VIX p90) – robustez",
    "so_local_sem_cauda": "Só crise local sem cauda de crise global – robustez (a)",
    "crise_local_fixa": "Lista fixa de eventos locais – robustez (b)",
    "choque_local_br": "Choque de 21 pregões no Ibovespa",
    "so_local_choque": "Choque local sem crise global – robustez (c)",
}

def episodios(indicador):
    return int(((indicador == 1) & (indicador.shift(1, fill_value=0) == 0)).sum())

contagem = pd.DataFrame([
    {"regime": coluna, "descrição": desc, "dias": int(base[coluna].sum()),
     "% da amostra": round(100 * base[coluna].mean(), 1), "episódios": episodios(base[coluna]),
     "maior episódio (dias)": int(base[coluna].groupby((base[coluna] != base[coluna].shift()).cumsum()).sum().max())}
    for coluna, desc in CONTAGEM.items()
])
contagem

,regime,descrição,dias,% da amostra,episódios,maior episódio (dias)
0,crise_global,VIX > 30,422,7.6,51,161
1,crise_global_p90,VIX > percentil 90,553,10.0,71,186
2,crise_local_br,Drawdown do Ibovespa ≤ −20%,729,13.2,59,222
3,crise_eua,Drawdown do S&P 500 ≤ −20%,314,5.7,20,225
4,crise_p1_eua,Crise EUA para a P1 (VIX > 30 ou drawdown do S...,524,9.5,57,234
5,normal,Normal,4686,84.8,81,482
6,so_global,Só crise global,111,2.0,39,10
7,so_local,Só crise local,418,7.6,66,60
8,ambas,Crise local e global,311,5.6,29,161
9,so_local_p90,Só crise local (VIX p90) – robustez,378,6.8,58,46


## 3. Ponto crítico da P4

A célula abaixo registra a contagem nas três definições e o número de episódios.

In [3]:
p4 = contagem.set_index("regime").loc[["so_local", "so_local_sem_cauda", "so_local_choque", "crise_local_fixa", "so_global"],
                                       ["dias", "episódios", "maior episódio (dias)"]]
p4

,dias,episódios,maior episódio (dias)
regime,,,
so_local,418,66,60
so_local_sem_cauda,296,36,53
so_local_choque,182,48,19
crise_local_fixa,84,4,21
so_global,111,39,10


## Saída

In [4]:
PASTA_RESULTADOS.mkdir(exist_ok=True)
contagem.to_csv(PASTA_RESULTADOS / "contagem_regimes.csv", index=False)
print("resultados/contagem_regimes.csv")

resultados/contagem_regimes.csv
